In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

In [ ]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")

In [ ]:
# Global Variables and Configs
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Image Constants
IMG_HEIGHT = 224
IMG_WIDTH = 407
IMG_CHANNELS = 3
IMAGE_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', 
    include_top=False, 
    pooling='avg', 
    input_shape=IMAGE_SHAPE
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [5]:
# Preprocessing Detection

def image_preprocessing(image):
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    image = image_preprocessing(data['images'])
    labels = data['labels']  # Array of integer Class IDs
    
    # 1. Track Humans: Pedestrians (1) OR People (2)
    is_human = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [1, 2]), axis=-1)
    human_count = tf.reduce_sum(tf.cast(is_human, tf.float32))
    
    # 2. Track Vehicles: Car (4), Vans (5), Trucks (6), Buses (9), Motorcycles (10)
    is_vehicle = tf.reduce_any(tf.equal(tf.expand_dims(labels, -1), [4, 5, 6, 9, 10]), axis=-1)
    vehicle_count = tf.reduce_sum(tf.cast(is_vehicle, tf.float32))
    
    counts_vector = tf.stack([human_count, vehicle_count], axis=0)
    return image, counts_vector

# Feature Stream Pipelines

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .cache()
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .cache()
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

In [ ]:
# Preprocessing City Labels

extract_train = train_stream.map(lambda img, _: img)
extract_val   = val_stream.map(lambda img, _: img)
extract_test  = test_stream.map(lambda img, _: img)

def classification_preprocessing(data_stream, pca_model=None, kmeans_model=None):
    features = GLOBAL_BACKBONE.predict(data_stream, verbose=0)

    if pca_model is None: # Training mode
        pca_model = PCA(n_components=50, random_state=42)
        kmeans_model = MiniBatchKMeans(n_clusters=14, random_state=42, n_init=5, batch_size=8192)
        labels = kmeans_model.fit_predict(pca_model.fit_transform(features))
        return tf.constant(labels, dtype=tf.int32), pca_model, kmeans_model
        
    # Inference mode
    labels = kmeans_model.predict(pca_model.transform(features))
    return tf.constant(labels, dtype=tf.int32)

# Training phase: captures the trained PCA and KMeans weights
tf_train_labels, trained_pca, trained_kmeans = classification_preprocessing(extract_train)

# Validation and Test phases: reuse those exact trained weights
tf_val_labels = classification_preprocessing(extract_val, pca_model=trained_pca, kmeans_model=trained_kmeans)
tf_test_labels = classification_preprocessing(extract_test, pca_model=trained_pca, kmeans_model=trained_kmeans)

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE)

In [7]:
# Adding Pipelines

def shape_structure_outputs(img_batch, count_batch, label_batch):
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, *IMAGE_SHAPE])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])   # [Batch_Size]
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [8]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(IMAGE_SHAPE), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.Conv2D(32, (3, 3), padding="same", activation='relu', kernel_initializer='he_normal')(shared_features)
    x_detect = layers.Flatten()(x_detect)
    
    x_detect = layers.Dense(128, activation=None, kernel_initializer='he_normal')(x_detect)
    x_detect = layers.BatchNormalization()(x_detect)
    x_detect = layers.LeakyReLU(negative_slope=0.1)(x_detect)
    
    # Final count output layer
    count_output = layers.Dense(2, activation='softplus', kernel_initializer='he_normal', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    
    protected_counts = layers.Lambda(lambda x: tf.stop_gradient(x))(count_output)

    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, protected_counts])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "huber"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "count_output": 0.1
        }
    )
    return model

china_model = build_multi_head_model()

history = china_model.fit(train_pipeline, validation_data=val_pipeline, epochs=3)

C:\Users\etito\AppData\Local\Temp\ipykernel_2604\770034392.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(



Epoch 1/3
    405/Unknown 184s 441ms/step - city_output_accuracy: 0.5198 - city_output_loss: 1.4506 - count_output_loss: 19.4693 - count_output_mae: 19.9404 - loss: 3.3976

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


405/405 ━━━━━━━━━━━━━━━━━━━━ 200s 482ms/step - city_output_accuracy: 0.6402 - city_output_loss: 1.0057 - count_output_loss: 16.5983 - count_output_mae: 17.1244 - loss: 2.6737 - val_city_output_accuracy: 0.8011 - val_city_output_loss: 0.5287 - val_count_output_loss: 13.6685 - val_count_output_mae: 14.6140 - val_loss: 1.9513
Epoch 2/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 192s 473ms/step - city_output_accuracy: 0.7875 - city_output_loss: 0.5703 - count_output_loss: 11.5765 - count_output_mae: 12.0968 - loss: 1.7323 - val_city_output_accuracy: 0.8741 - val_city_output_loss: 0.3376 - val_count_output_loss: 12.1982 - val_count_output_mae: 13.1419 - val_loss: 1.6100
Epoch 3/3
405/405 ━━━━━━━━━━━━━━━━━━━━ 193s 476ms/step - city_output_accuracy: 0.8218 - city_output_loss: 0.4717 - count_output_loss: 10.1889 - count_output_mae: 10.7009 - loss: 1.4946 - val_city_output_accuracy: 0.8376 - val_city_output_loss: 0.3833 - val_count_output_loss: 12.4381 - val_count_output_mae: 13.4117 - val_loss: 1.6839


In [10]:
# Testing Model

results = china_model.evaluate(test_pipeline, return_dict=True)
print(results)

for test_images, test_targets in test_pipeline.take(1):
    city_preds, count_preds = china_model.predict(test_images, verbose=0)
    
    predicted_city_indices = np.argmax(city_preds, axis=-1)
    true_city_indices = np.array(test_targets["city_output"]).astype(int).flatten()
    true_counts = np.array(test_targets["count_output"])
    
    for i in range(min(5, len(true_city_indices))):
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"  [City]  Predicted: {pred_label:<12} | True: {true_label}")
        print(f"  [Human] Predicted: {count_preds[i][0]:<12.2f} | True: {true_counts[i][0]:.1f}")
        print(f"  [Car]   Predicted: {count_preds[i][1]:<12.2f} | True: {true_counts[i][1]:.1f}")

101/101 ━━━━━━━━━━━━━━━━━━━━ 63s 627ms/step - city_output_accuracy: 0.8534 - city_output_loss: 0.3587 - count_output_loss: 13.7309 - count_output_mae: 14.3537 - loss: 1.7486
{'city_output_accuracy': 0.8534161448478699, 'city_output_loss': 0.35872912406921387, 'count_output_loss': 13.730925559997559, 'count_output_mae': 14.353679656982422, 'loss': 1.748647689819336}

Sample 1:
  [City]  Predicted: jinchang     | True: jinchang
  [Human] Predicted: 11.71        | True: 0.0
  [Car]   Predicted: 8.11         | True: 15.0

Sample 2:
  [City]  Predicted: guangzhou    | True: guangzhou
  [Human] Predicted: 15.73        | True: 0.0
  [Car]   Predicted: 43.26        | True: 29.0

Sample 3:
  [City]  Predicted: suzhou       | True: suzhou
  [Human] Predicted: 24.60        | True: 54.0
  [Car]   Predicted: 38.57        | True: 48.0

Sample 4:
  [City]  Predicted: hong-kong    | True: hong-kong
  [Human] Predicted: 23.24        | True: 12.0
  [Car]   Predicted: 38.06        | True: 33.0

Sample 5: